# 🐾 Pet Facial Expression Classification
Transfer learning (MobileNetV2) on the Kaggle **Pets Facial Expression Dataset** (`anshtanwar/pets-facial-expression-dataset`).

**Before running:** `Runtime → Change runtime type → T4 GPU`.

In [ ]:
import os, random, numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

## 1. Download the dataset

In [ ]:
import kagglehub

path = kagglehub.dataset_download("anshtanwar/pets-facial-expression-dataset")
print("Path to dataset files:", path)

## 2. Locate train / valid / test folders

In [ ]:
def find_splits(root):
    names = {"train": "train", "valid": "valid", "validation": "valid", "val": "valid", "test": "test"}
    found = {}
    for dp, dirs, _ in os.walk(root):
        for d in dirs:
            key = d.lower()
            if key in names and names[key] not in found:
                sub = [x for x in os.listdir(os.path.join(dp, d)) if os.path.isdir(os.path.join(dp, d, x))]
                if sub:  # must contain class folders
                    found[names[key]] = os.path.join(dp, d)
    return found

splits = find_splits(path)
print(splits)
for k, v in splits.items():
    print(k, {c: len(os.listdir(os.path.join(v, c))) for c in sorted(os.listdir(v))})

## 3. Load data

In [ ]:
IMG_SIZE = (224, 224)
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

def load(split, shuffle):
    return keras.utils.image_dataset_from_directory(
        splits[split], image_size=IMG_SIZE, batch_size=BATCH,
        label_mode="int", shuffle=shuffle, seed=SEED)

train_ds = load("train", True)
val_ds   = load("valid", False) if "valid" in splits else None
test_ds  = load("test", False) if "test" in splits else None

# Fallback if the dataset has no separate validation folder
if val_ds is None:
    train_ds = keras.utils.image_dataset_from_directory(
        splits["train"], validation_split=0.2, subset="training", seed=SEED,
        image_size=IMG_SIZE, batch_size=BATCH)
    val_ds = keras.utils.image_dataset_from_directory(
        splits["train"], validation_split=0.2, subset="validation", seed=SEED,
        image_size=IMG_SIZE, batch_size=BATCH)
if test_ds is None:
    test_ds = val_ds

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("Classes:", class_names)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

## 4. Look at the data

In [ ]:
plt.figure(figsize=(12, 8))
for images, labels in train_ds.take(1):
    for i in range(min(12, len(images))):
        plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.tight_layout(); plt.show()

# Class balance (used for class weights)
counts = np.zeros(NUM_CLASSES)
for _, y in train_ds:
    for v in y.numpy(): counts[v] += 1
print(dict(zip(class_names, counts.astype(int))))
total = counts.sum()
class_weight = {i: total / (NUM_CLASSES * c) for i, c in enumerate(counts)}
print("Class weights:", class_weight)

## 5. Build the model (MobileNetV2 transfer learning)

In [ ]:
data_aug = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
], name="augmentation")

base = keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_aug(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 6. Train (stage 1: frozen backbone)

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
    keras.callbacks.ReduceLROnPlateau(factor=0.3, patience=2, verbose=1),
]
h1 = model.fit(train_ds, validation_data=val_ds, epochs=10,
               class_weight=class_weight, callbacks=callbacks)

## 7. Fine-tune (stage 2: unfreeze top layers)

In [ ]:
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=10,
               class_weight=class_weight, callbacks=callbacks)

## 8. Training curves

In [ ]:
acc  = h1.history["accuracy"] + h2.history["accuracy"]
vacc = h1.history["val_accuracy"] + h2.history["val_accuracy"]
loss = h1.history["loss"] + h2.history["loss"]
vloss = h1.history["val_loss"] + h2.history["val_loss"]
split_at = len(h1.history["accuracy"]) - 1

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, tr, va, t in zip(ax, [acc, loss], [vacc, vloss], ["Accuracy", "Loss"]):
    a.plot(tr, label="train"); a.plot(va, label="val")
    a.axvline(split_at, color="gray", ls="--", label="fine-tune start")
    a.set_title(t); a.legend()
plt.show()

## 9. Evaluate on the test set

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.3f}")

y_true, y_pred = [], []
for x, y in test_ds:
    y_true += list(y.numpy())
    y_pred += list(np.argmax(model.predict(x, verbose=0), axis=1))

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix")
plt.show()

## 10. Predict on your own pet photo

In [ ]:
from google.colab import files
from PIL import Image

def predict_image(img_path):
    img = Image.open(img_path).convert("RGB").resize(IMG_SIZE)
    arr = np.expand_dims(np.array(img), 0).astype("float32")
    probs = model.predict(arr, verbose=0)[0]
    plt.imshow(img); plt.axis("off")
    plt.title(f"{class_names[probs.argmax()]} ({probs.max():.0%})")
    plt.show()
    for c, p in sorted(zip(class_names, probs), key=lambda t: -t[1]):
        print(f"{c:>8}: {p:.1%}")

uploaded = files.upload()
for name in uploaded:
    predict_image(name)

## 11. Save the model

In [ ]:
model.save("pet_expression_model.keras")
files.download("pet_expression_model.keras")